<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 03. Fechas y Errores de Escritura — Traducir fechas enredadas y corregir typos
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/05%20-%20Data%20Preparation/Para%20Dummies/03_Fechas_y_Datos_Inconsistentes_Data_Preparation_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno aborda los dos problemas de calidad de texto más comunes en datos reales:

1. **Fechas en formatos inconsistentes** — `pd.to_datetime()` y extracción de partes de la fecha.
2. **Errores tipográficos en texto** — Fuzzy Matching (búsqueda aproximada) con `fuzzywuzzy`.

---
## PARTE A: Fechas — El caos de los formatos 📅

### 1. El problema: todos dicen lo mismo de distinta forma

Piensa en cómo se puede escribir la misma fecha:

| Lo que significa | Cómo lo escriben | ¿Python lo entiende? |
|---|---|---|
| 31 de diciembre de 2024 | `2024-12-31` | ✅ Sí (ISO 8601) |
| 31 de diciembre de 2024 | `31/12/2024` | ⚠️ A veces |
| 31 de diciembre de 2024 | `Dec 31, 2024` | ⚠️ A veces |
| 31 de diciembre de 2024 | `12-31-24` | ❌ Confunde mes y día |

Si una columna tiene fechas en 3 formatos diferentes, Python la leerá como texto (`object`) y no podrá calcular diferencias de días, extraer el mes, etc.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ejemplo de fechas en múltiples formatos (como las encontrarías en el mundo real)
fechas_caos = pd.Series([
    '2024-01-15',
    '15/01/2024',
    'January 15, 2024',
    '2024.01.15',
    '15-Jan-2024',
    None
])

print("📅 Fechas en caos (tipo actual):")
print(fechas_caos)
print(f"\nTipo de dato actual: {fechas_caos.dtype}  ← Python las trata como texto, no como fechas")

In [ ]:
# pd.to_datetime con errors='coerce' es el truco: convierte lo que puede, pone NaT (Not a Time) en lo que no puede
fechas_limpias = pd.to_datetime(fechas_caos, dayfirst=True, errors='coerce')

print("✅ Fechas convertidas al formato estándar:")
print(fechas_limpias)
print(f"\nTipo de dato ahora: {fechas_limpias.dtype}  ← datetime64, Python ahora las entiende como fechas")
print(f"NaT (fecha inválida o nula): {fechas_limpias.isnull().sum()} fechas")

### 2. Extraer partes de la fecha — El verdadero poder de datetime

Una vez que tienes las fechas en formato correcto, puedes extraer año, mes, día, día de la semana, etc. Esto es muy útil para análisis temporales.

> 💡 **Ejemplo real:** Con una columna de fecha de compra puedes crear automáticamente: `mes_compra`, `dia_semana_compra`, `es_fin_de_semana`, `trimestre` — variables muy útiles para predecir comportamiento de clientes.

In [ ]:
import os, urllib.parse, urllib.request

def load_dataset(filename, module_name="05 - Data Preparation"):
    candidates = [f"data/{filename}", f"../{module_name}/data/{filename}", f"{module_name}/data/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    target_path = f"data/{filename}"
    url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{urllib.parse.quote(module_name)}/data/{urllib.parse.quote(filename)}"
    urllib.request.urlretrieve(url, target_path)
    print(f"✅ Dataset '{filename}' descargado.")
    return target_path

df = pd.read_csv(load_dataset('landslide-events.csv'), low_memory=False)
print(f"Dataset cargado: {df.shape[0]} eventos de deslizamiento")

# Buscar la columna de fecha
col_fecha = [c for c in df.columns if 'date' in c.lower() or 'fecha' in c.lower()][0]
print(f"Columna de fecha encontrada: '{col_fecha}'")
print(f"Tipo actual: {df[col_fecha].dtype}")
print(df[col_fecha].head())

In [ ]:
# Convertir la columna de fecha y extraer componentes temporales
df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

df['año']             = df[col_fecha].dt.year
df['mes']             = df[col_fecha].dt.month
df['dia']             = df[col_fecha].dt.day
df['dia_semana']      = df[col_fecha].dt.day_name()
df['es_fin_semana']   = df[col_fecha].dt.dayofweek >= 5
df['trimestre']       = df[col_fecha].dt.quarter

print("✅ Nuevas columnas extraídas de la fecha:")
print(df[[col_fecha, 'año', 'mes', 'dia', 'dia_semana', 'es_fin_semana', 'trimestre']].head(8).to_string(index=False))

In [ ]:
# Análisis temporal: ¿en qué mes ocurren más deslizamientos?
plt.figure(figsize=(12, 4))
meses_nombres = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
conteo_mes = df['mes'].value_counts().sort_index()
plt.bar(conteo_mes.index, conteo_mes.values, color='#f59e0b', edgecolor='white', linewidth=1.5)
plt.xticks(range(1,13), meses_nombres)
plt.title('Deslizamientos de tierra por mes\n(posible relación con temporadas de lluvia)', fontweight='bold', fontsize=13)
plt.ylabel('Número de eventos')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\n💡 ¿Ves un patrón? Este análisis solo fue posible porque convertimos la fecha correctamente.")

---
## PARTE B: Errores de Escritura — El corrector ortográfico inteligente ✏️

### 3. El problema: "Colombia" en 10 versiones diferentes

En bases de datos del mundo real, especialmente las llenadas manualmente, es común encontrar:

```
'Colombia', 'COLOMBIA', 'Colombiia', 'Kolumbia', 'colombia', 'Cloombia'
```

Para Python, **todas esas son categorías diferentes**. Si haces un conteo, obtienes 6 países en lugar de 1.

La solución es el **Fuzzy Matching** (coincidencia aproximada): un algoritmo que mide qué tan _parecidas_ son dos palabras aunque no sean idénticas. Funciona como el corrector de tu celular, pero para datos.

In [ ]:
# Instalar fuzzywuzzy si no está disponible
try:
    from fuzzywuzzy import fuzz, process
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fuzzywuzzy", "python-Levenshtein", "-q"])
    from fuzzywuzzy import fuzz, process

print("✅ fuzzywuzzy disponible.")

# Demostración: similitud entre palabras
pares = [
    ('Colombia', 'colombia'),
    ('Colombia', 'Colombiia'),
    ('Colombia', 'Kolumbia'),
    ('Colombia', 'México'),
    ('Colombia', 'Brazil'),
]

print("\n🔍 Similitud entre 'Colombia' y otras variantes:")
print(f"{'Par':<30} {'Similitud':>12}")
print("-" * 44)
for a, b in pares:
    score = fuzz.ratio(a, b)
    barra = '█' * (score // 5)
    print(f"'{a}' vs '{b}':{' '*(15-len(b))} {score:>3}%  {barra}")

In [ ]:
# Aplicación práctica: estandarizar nombres de países con errores
paises_sucios = [
    'Colombiia', 'COLOMBIA', 'Colombia',
    'Brazilll', 'Brazil', 'Brasil',
    'Mexiko', 'Mexico', 'MEXICO',
    'Argntina', 'Argentina',
]

# Lista de nombres canónicos (los correctos)
canonicos = ['Colombia', 'Brazil', 'Mexico', 'Argentina', 'Peru', 'Chile']

def corregir_nombre(nombre_sucio, lista_canonicos, umbral=70):
    """Corrige un nombre buscando el más similar en la lista canónica."""
    mejor, puntaje = process.extractOne(nombre_sucio, lista_canonicos)
    if puntaje >= umbral:
        return mejor
    return nombre_sucio  # si no hay suficiente similitud, deja el original

print("\n✏️ Corrección automática de nombres de países:")
print(f"{'Nombre sucio':<15} → {'Corregido':<15} {'Similitud'}")
print("-" * 50)
for sucio in paises_sucios:
    correcto, puntaje = process.extractOne(sucio, canonicos)
    icon = '✅' if puntaje >= 70 else '⚠️'
    print(f"{icon} '{sucio}'{'':>{14-len(sucio)}} → '{correcto}'{'':>{14-len(correcto)}} ({puntaje}%)")

In [ ]:
# Aplicar la corrección a una columna completa del dataset real
df_pk = pd.read_csv(load_dataset('pakistan_intellectual_capital.csv'), low_memory=False)

# Buscar columna de país
col_pais = [c for c in df_pk.columns if 'country' in c.lower()][0]
print(f"Columna de país: '{col_pais}'")
print(f"\nValores únicos originales (muestra):")
print(df_pk[col_pais].value_counts().head(15).to_string())

In [ ]:
# Estandarizar los países más frecuentes
paises_frecuentes = df_pk[col_pais].value_counts().head(20).index.tolist()

# Lista canónica de países de referencia
canonicos_pk = ['Pakistan', 'United Kingdom', 'United States', 'Australia',
                'China', 'Germany', 'Canada', 'France', 'Malaysia', 'Saudi Arabia']

# Crear mapa de correcciones
mapa_correcciones = {}
print("\n🗺️ Mapa de correcciones generado por Fuzzy Matching:")
for pais in paises_frecuentes:
    if pd.notna(pais):
        resultado, puntaje = process.extractOne(str(pais), canonicos_pk)
        if puntaje >= 60:
            mapa_correcciones[pais] = resultado
            if pais != resultado:
                print(f"  '{pais}' → '{resultado}' ({puntaje}%)")

# Aplicar correcciones
df_pk[col_pais + '_limpio'] = df_pk[col_pais].map(mapa_correcciones).fillna(df_pk[col_pais])

print(f"\n✅ Correcciones aplicadas a la columna '{col_pais}'.")

---
## Resumen final del módulo Data Preparation 🎓

| Problema | Herramienta | Cuándo usarla |
|---|---|---|
| **Valores faltantes numéricos** | `SimpleImputer(strategy='median')` | La columna tiene outliers |
| **Valores faltantes categóricos** | `SimpleImputer(strategy='most_frequent')` | Variables de texto |
| **Escalas diferentes** | `MinMaxScaler` o `StandardScaler` | Antes de KNN, SVM, redes neuronales |
| **Outliers + escalado** | `RobustScaler` | Cuando hay outliers marcados |
| **Fechas en múltiples formatos** | `pd.to_datetime(errors='coerce')` | Siempre que haya columnas de fecha |
| **Errores tipográficos en texto** | `fuzzywuzzy.process.extractOne()` | Nombres de países, ciudades, productos |

> 🏁 **¡Completaste el módulo 05!** Ahora tus datos están listos para el módulo 06 de Feature Engineering y los modelos de Machine Learning.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>